---

# BirdCLEF+ 2026 - exp003 Submission

**CPU Notebook / 推論専用 / 90分以内**

### exp002 からの改善点
- **Perch v2** (Google 凍結特徴抽出器) + embedding probe をベースに採用 (ref: 0.910 スコアノートブック)
- **exp002 EfficientNet SED** モデルとのアンサンブル (Perch 0.7 + EfficientNet 0.3)

### Kaggle Notebook の Input に追加するもの
- `birdclef-2026` … コンペデータ
- `jaejohn/perch-meta` … Perch キャッシュ (full_perch_meta.parquet + full_perch_arrays.npz)
- `google/bird-vocalization-classifier (perch_v2_cpu)` … Perch v2 SavedModel
- `kdmitrie/bc26-tensorflow-2-20-0` … TF 2.20.0 ホイール
- `birdclef2026-exp002-weights` … exp002 EfficientNet 学習済み重み (best_fold0.pth)

## Install

In [ ]:
import subprocess, sys
from pathlib import Path

_WHL = Path('/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel')
if not _WHL.exists():
    msg = 'Wheel directory not found: ' + str(_WHL) + '. Add kdmitrie/bc26-tensorflow-2-20-0 as a Notebook input (not Dataset).'
    raise RuntimeError(msg)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    str(_WHL / 'tensorboard-2.20.0-py3-none-any.whl')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    str(_WHL / 'tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl')], check=True)
print('Installed TF 2.20.0 from', _WHL)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
print('Installed timm.')

## Libraries

In [ ]:
import gc
import json
import os
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import timm
from scipy.ndimage import convolve1d
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # force CPU
tf.experimental.numpy.experimental_enable_numpy_behavior()

print('TensorFlow :', tf.__version__)
print('PyTorch    :', torch.__version__)
print('NumPy      :', np.__version__)
print('Pandas     :', pd.__version__)

## Settings

In [ ]:
class Settings:
    # Mode ------------------------------------------------------------------
    MODE = 'submit'  # 'submit' or 'train'
    SEED = 42

    # Competition paths -----------------------------------------------------
    _KAGGLE_BASE1 = Path('/kaggle/input/birdclef-2026')
    _KAGGLE_BASE2 = Path('/kaggle/input/competitions/birdclef-2026')
    _LOCAL_BASE   = Path('../dataset')
    BASE = (
        _KAGGLE_BASE1 if _KAGGLE_BASE1.exists() else
        _KAGGLE_BASE2 if _KAGGLE_BASE2.exists() else
        _LOCAL_BASE
    )

    # Perch v2 SavedModel ---------------------------------------------------
    MODEL_DIR = Path(
        '/kaggle/input/models/google/bird-vocalization-classifier'
        '/tensorflow2/perch_v2_cpu/1'
    )

    # Perch cache (full_perch_meta.parquet + full_perch_arrays.npz)
    _CACHE_CANDIDATES = [
        Path('/kaggle/input/datasets/jaejohn/perch-meta'),
        Path('../input'),
        Path('/kaggle/working/cache'),
    ]
    CACHE_DIR = next(
        (
            d for d in _CACHE_CANDIDATES
            if (d / 'full_perch_meta.parquet').exists()
            and (d / 'full_perch_arrays.npz').exists()
        ),
        None,
    )
    CACHE_EXISTS = CACHE_DIR is not None
    WORK_CACHE_DIR = Path('/kaggle/working/cache')

    # Audio (Perch) ---------------------------------------------------------
    SR             = 32_000
    WINDOW_SEC     = 5
    WINDOW_SAMPLES = SR * WINDOW_SEC
    FILE_SAMPLES   = 60 * SR
    N_WINDOWS      = 12
    BATCH_FILES    = 16
    DRYRUN_N_FILES = 20

    # Prior fusion (frozen from OOF tuning) ---------------------------------
    LAMBDA_EVENT         = 0.4
    LAMBDA_TEXTURE       = 1.0
    LAMBDA_PROXY_TEXTURE = 0.8
    SMOOTH_TEXTURE_ALPHA = 0.35

    # Embedding probe (frozen from OOF tuning) ------------------------------
    PROBE_PCA_DIM = 32
    PROBE_MIN_POS = 8
    PROBE_C       = 0.25
    PROBE_ALPHA   = 0.40

    # exp002 EfficientNet ---------------------------------------------------
    _EFF_WEIGHT_CANDIDATES = [
        Path('/kaggle/input/birdclef2026-exp002-weights/best_fold0.pth'),
        *Path('/kaggle/input').glob('**/best_fold0.pth'),
    ]
    EFF_WEIGHT_PATH = next(
        (p for p in _EFF_WEIGHT_CANDIDATES if p.exists()), None
    )
    EFF_SAMPLE_RATE    = 32000
    EFF_N_SAMPLES      = 32000 * 10       # 10秒ウィンドウ
    EFF_N_MELS         = 128
    EFF_N_FFT          = 1024
    EFF_HOP_LENGTH     = 320
    EFF_FMIN           = 20
    EFF_FMAX           = 16000
    EFF_BATCH_SIZE     = 16
    EFF_OVERLAP_OFFSETS = [-2.5, 0.0, 2.5]  # 重複推論オフセット（秒）

    # Ensemble --------------------------------------------------------------
    ENSEMBLE_W_PERCH = 0.7  # Perch の重み
    ENSEMBLE_W_EFF   = 0.3  # EfficientNet の重み


CFG = Settings()
CFG.WORK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODE             : {CFG.MODE}')
print(f'BASE             : {CFG.BASE}')
print(f'CACHE_EXISTS     : {CFG.CACHE_EXISTS}')
print(f'CACHE_DIR        : {CFG.CACHE_DIR}')
print(f'EFF_WEIGHT_PATH  : {CFG.EFF_WEIGHT_PATH}')
print(f'EFF weight exists: {CFG.EFF_WEIGHT_PATH is not None and CFG.EFF_WEIGHT_PATH.exists()}')

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

---

# Data Loading

In [ ]:
taxonomy        = pd.read_csv(CFG.BASE / 'taxonomy.csv')
train_meta      = pd.read_csv(CFG.BASE / 'train.csv')
soundscape_raw  = pd.read_csv(CFG.BASE / 'train_soundscapes_labels.csv')
sample_sub      = pd.read_csv(CFG.BASE / 'sample_submission.csv')

soundscape_lbls = soundscape_raw.drop_duplicates().reset_index(drop=True)

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

print(f'taxonomy species     : {len(taxonomy)}')
print(f'train recordings     : {len(train_meta):,}')
print(f'soundscape rows      : {len(soundscape_lbls):,} unique '
      f'(dropped {len(soundscape_raw) - len(soundscape_lbls):,} duplicates)')
print(f'submission classes   : {N_CLASSES}')

In [ ]:
FNAME_RE = re.compile(
    r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg'
)

def parse_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(';') if t.strip()]

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_labels(x)))

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {'site': None, 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

sc_clean = (
    soundscape_lbls
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id']  = (
    sc_clean['filename'].str.replace('.ogg', '', regex=False)
    + '_' + sc_clean['end_sec'].astype(str)
)
meta_cols = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean  = pd.concat([sc_clean, meta_cols], axis=1)

wpf        = sc_clean.groupby('filename').size()
full_files = sorted(wpf[wpf == CFG.N_WINDOWS].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean['label_list']):
    for lbl in labels:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

full_truth = (
    sc_clean[sc_clean['file_fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)
Y_FULL_TRUTH = Y_SC[full_truth['index'].to_numpy()]

print(f'Fully-labeled files : {len(full_files)}')
print(f'Trusted windows     : {len(full_truth)}')
print(f'Active classes      : {int((Y_FULL_TRUTH.sum(axis=0) > 0).sum())}')

---

# Perch Label Mapping

In [ ]:
print('Loading Perch model...')
birdclassifier = tf.saved_model.load(str(CFG.MODEL_DIR))
infer_fn       = birdclassifier.signatures['serving_default']
print('Perch loaded.')

bc_labels = (
    pd.read_csv(CFG.MODEL_DIR / 'assets' / 'labels.csv')
    .reset_index()
    .rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'})
)
NO_LABEL_INDEX = len(bc_labels)

taxonomy_ = taxonomy.copy()
taxonomy_['scientific_name'] = taxonomy_['scientific_name'].astype(str)
mapping = taxonomy_.merge(
    bc_labels[['scientific_name', 'bc_index']],
    on='scientific_name', how='left'
)
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc   = mapping.set_index('primary_label')['bc_index']
BC_INDICES    = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK       = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS        = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS      = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f'Mapped   : {MAPPED_MASK.sum()} / {N_CLASSES}')
print(f'Unmapped : {(~MAPPED_MASK).sum()}')

In [ ]:
CLASS_NAME_MAP = taxonomy_.set_index('primary_label')['class_name'].to_dict()
TEXTURE_TAXA   = {'Amphibia', 'Insecta'}
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
    dtype=np.int32
)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA],
    dtype=np.int32
)

idx_mapped_active_texture  = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event    = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event   = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES], dtype=np.int32
)

print(f'Active texture classes : {len(idx_active_texture)}')
print(f'Active event classes   : {len(idx_active_event)}')
print(f'Unmapped inactive      : {len(idx_unmapped_inactive)}')

In [ ]:
unmapped_df = mapping[mapping['bc_index'] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df['primary_label'].astype(str).str.contains('son', na=False)
].copy()

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    genus = str(row['scientific_name']).split()[0]
    hits  = bc_labels[
        bc_labels['scientific_name'].str.match(rf'^{re.escape(genus)}\s', na=False)
    ]
    if len(hits) > 0:
        proxy_map[str(row['primary_label'])] = hits['bc_index'].astype(int).tolist()

SELECTED_PROXY_TARGETS   = sorted([t for t in proxy_map if CLASS_NAME_MAP.get(t) == 'Amphibia'])
selected_proxy_pos       = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {
    label_to_idx[t]: np.array(proxy_map[t], dtype=np.int32) for t in SELECTED_PROXY_TARGETS
}

idx_selected_proxy_active_texture  = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event   = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f'Frog proxy targets : {SELECTED_PROXY_TARGETS}')

---

# Perch Inference

In [ ]:
def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < CFG.FILE_SAMPLES:
        y = np.pad(y, (0, CFG.FILE_SAMPLES - len(y)))
    return y[:CFG.FILE_SAMPLES]


def infer_perch_batch(paths, verbose=True):
    paths   = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows  = n_files * CFG.N_WINDOWS

    row_ids    = np.empty(n_rows, dtype=object)
    filenames  = np.empty(n_rows, dtype=object)
    sites      = np.empty(n_rows, dtype=object)
    hours      = np.empty(n_rows, dtype=np.int16)
    scores     = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536),      dtype=np.float32)

    write_row = 0
    itr = tqdm(range(0, n_files, CFG.BATCH_FILES), desc='Perch', disable=not verbose)

    for start in itr:
        batch  = paths[start:start + CFG.BATCH_FILES]
        bn     = len(batch)
        x      = np.empty((bn * CFG.N_WINDOWS, CFG.WINDOW_SAMPLES), dtype=np.float32)
        bstart = write_row

        for bi, path in enumerate(batch):
            audio = read_soundscape_60s(path)
            x[bi * CFG.N_WINDOWS:(bi + 1) * CFG.N_WINDOWS] = audio.reshape(
                CFG.N_WINDOWS, CFG.WINDOW_SAMPLES
            )
            meta = parse_soundscape_filename(path.name)
            row_ids[write_row:write_row + CFG.N_WINDOWS]   = [
                f'{path.stem}_{t}' for t in range(5, 65, 5)
            ]
            filenames[write_row:write_row + CFG.N_WINDOWS] = path.name
            sites[write_row:write_row + CFG.N_WINDOWS]     = meta['site']
            hours[write_row:write_row + CFG.N_WINDOWS]     = meta['hour_utc']
            write_row += CFG.N_WINDOWS

        out    = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = out['label'].numpy().astype(np.float32)
        emb    = out['embedding'].numpy().astype(np.float32)

        scores[bstart:write_row, MAPPED_POS] = logits[:write_row - bstart, MAPPED_BC_INDICES]
        embeddings[bstart:write_row]          = emb

        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            scores[bstart:write_row, pos] = logits[:write_row - bstart, bc_idx_arr].max(axis=1)

        del x, out, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        'row_id': row_ids, 'filename': filenames,
        'site': sites, 'hour_utc': hours,
    })
    return meta_df, scores, embeddings

In [ ]:
if CFG.CACHE_EXISTS:
    print(f'Loading Perch cache from: {CFG.CACHE_DIR}')
    meta_full       = pd.read_parquet(CFG.CACHE_DIR / 'full_perch_meta.parquet')
    arr             = np.load(CFG.CACHE_DIR / 'full_perch_arrays.npz')
    scores_full_raw = arr['scores_full_raw'].astype(np.float32)
    emb_full        = arr['emb_full'].astype(np.float32)
else:
    print('No cache found. Running Perch on fully-labeled training soundscapes...')
    full_paths = [CFG.BASE / 'train_soundscapes' / fn for fn in full_files]
    meta_full, scores_full_raw, emb_full = infer_perch_batch(full_paths)
    meta_full.to_parquet(CFG.WORK_CACHE_DIR / 'full_perch_meta.parquet', index=False)
    np.savez_compressed(
        CFG.WORK_CACHE_DIR / 'full_perch_arrays.npz',
        scores_full_raw=scores_full_raw,
        emb_full=emb_full,
    )
    print(f'Cache saved to {CFG.WORK_CACHE_DIR}')

full_truth_aligned = (
    full_truth.set_index('row_id')
    .loc[meta_full['row_id']]
    .reset_index(drop=False)
)
Y_FULL = Y_SC[full_truth_aligned['index'].to_numpy()]

print(f'scores_full_raw : {scores_full_raw.shape}  {scores_full_raw.dtype}')
print(f'emb_full        : {emb_full.shape}  {emb_full.dtype}')
print(f'Y_FULL          : {Y_FULL.shape}')

---

# Prior Fusion

In [ ]:
def fit_prior_tables(prior_df, Y_prior):
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique())
    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique())

    site_to_i, site_n, site_p = {}, [], []
    for s in site_keys:
        mask = prior_df['site'].astype(str).values == s
        site_to_i[s] = len(site_n)
        site_n.append(mask.sum())
        site_p.append(Y_prior[mask].mean(axis=0))
    site_n = np.array(site_n, dtype=np.float32)
    site_p = np.stack(site_p).astype(np.float32) if site_p else np.zeros((0, Y_prior.shape[1]), np.float32)

    hour_to_i, hour_n, hour_p = {}, [], []
    for h in hour_keys:
        mask = prior_df['hour_utc'].astype(int).values == h
        hour_to_i[h] = len(hour_n)
        hour_n.append(mask.sum())
        hour_p.append(Y_prior[mask].mean(axis=0))
    hour_n = np.array(hour_n, dtype=np.float32)
    hour_p = np.stack(hour_p).astype(np.float32) if hour_p else np.zeros((0, Y_prior.shape[1]), np.float32)

    sh_to_i, sh_n_list, sh_p_list = {}, [], []
    for (s, h), idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))
    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, Y_prior.shape[1]), np.float32)

    return dict(
        global_p=global_p,
        site_to_i=site_to_i, site_n=site_n, site_p=site_p,
        hour_to_i=hour_to_i, hour_n=hour_n, hour_p=hour_p,
        sh_to_i=sh_to_i,     sh_n=sh_n,     sh_p=sh_p,
    )

In [ ]:
def prior_logits(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables['global_p'][None, :], n, axis=0).astype(np.float32, copy=True)

    si  = np.fromiter((tables['site_to_i'].get(str(s), -1) for s in sites), np.int32, n)
    hi  = np.fromiter(
        (tables['hour_to_i'].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
        np.int32, n
    )
    shi = np.fromiter(
        (tables['sh_to_i'].get((str(s), int(h)), -1) if int(h) >= 0 else -1
         for s, h in zip(sites, hours)),
        np.int32, n
    )

    valid = hi >= 0
    if valid.any():
        nh = tables['hour_n'][hi[valid]][:, None]
        p[valid] = nh / (nh + 8.0) * tables['hour_p'][hi[valid]] + (1.0 - nh / (nh + 8.0)) * p[valid]

    valid = si >= 0
    if valid.any():
        ns = tables['site_n'][si[valid]][:, None]
        p[valid] = ns / (ns + 8.0) * tables['site_p'][si[valid]] + (1.0 - ns / (ns + 8.0)) * p[valid]

    valid = shi >= 0
    if valid.any():
        nsh = tables['sh_n'][shi[valid]][:, None]
        p[valid] = nsh / (nsh + 4.0) * tables['sh_p'][shi[valid]] + (1.0 - nsh / (nsh + 4.0)) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)


def smooth_cols(scores, cols, alpha=0.35):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s    = scores.copy()
    view = s.reshape(-1, CFG.N_WINDOWS, s.shape[1])
    x    = view[:, :, cols]
    prev = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    nxt  = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev + nxt)
    return s


def fuse_scores(base, sites, hours, tables):
    scores = base.copy()
    prior  = prior_logits(sites, hours, tables)

    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += CFG.LAMBDA_EVENT * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += CFG.LAMBDA_TEXTURE * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += (
            CFG.LAMBDA_PROXY_TEXTURE * prior[:, idx_selected_proxy_active_texture]
        )
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = (
            CFG.LAMBDA_EVENT * prior[:, idx_selected_prioronly_active_event]
        )
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = (
            CFG.LAMBDA_TEXTURE * prior[:, idx_selected_prioronly_active_texture]
        )
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols(scores, idx_active_texture, alpha=CFG.SMOOTH_TEXTURE_ALPHA)
    return scores.astype(np.float32), prior

---

# Out-of-Fold Meta-features

In [ ]:
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')


gkf    = GroupKFold(n_splits=5)
groups = meta_full['site'].to_numpy()

oof_base  = np.zeros_like(scores_full_raw, dtype=np.float32)
oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)

for _, va_idx in tqdm(list(gkf.split(scores_full_raw, groups=groups)), desc='OOF folds'):
    va_idx    = np.sort(va_idx)
    val_sites = set(meta_full.iloc[va_idx]['site'].tolist())
    prior_m   = ~sc_clean['site'].isin(val_sites).values
    tables    = fit_prior_tables(
        sc_clean.loc[prior_m].reset_index(drop=True), Y_SC[prior_m]
    )
    oof_base[va_idx], oof_prior[va_idx] = fuse_scores(
        scores_full_raw[va_idx],
        meta_full.iloc[va_idx]['site'].to_numpy(),
        meta_full.iloc[va_idx]['hour_utc'].to_numpy(),
        tables,
    )

print(f'OOF baseline AUC (prior fusion only): {macro_auc(Y_FULL, oof_base):.6f}')

---

# Embedding Probes

In [ ]:
def seq_features_1d(v):
    x    = v.reshape(-1, CFG.N_WINDOWS)
    prev = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    nxt  = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    return prev, nxt, np.repeat(x.mean(1), CFG.N_WINDOWS), np.repeat(x.max(1), CFG.N_WINDOWS)


def build_class_features(Z, raw_col, prior_col, base_col):
    p, n, m, mx = seq_features_1d(base_col)
    return np.concatenate(
        [Z, raw_col[:, None], prior_col[:, None], base_col[:, None],
         p[:, None], n[:, None], m[:, None], mx[:, None]],
        axis=1
    ).astype(np.float32)

In [ ]:
emb_scaler = StandardScaler()
emb_scaled = emb_scaler.fit_transform(emb_full)

n_comp = min(CFG.PROBE_PCA_DIM, emb_scaled.shape[0] - 1, emb_scaled.shape[1])
emb_pca = PCA(n_components=n_comp)
Z_FULL  = emb_pca.fit_transform(emb_scaled).astype(np.float32)

print(f'PCA components  : {n_comp}')
print(f'Explained var   : {emb_pca.explained_variance_ratio_.sum():.4f}')

In [ ]:
if CFG.MODE == 'train':
    param_grid = [
        {'pca_dim': 32, 'min_pos': 8,  'C': 0.25, 'alpha': 0.40},
        {'pca_dim': 64, 'min_pos': 8,  'C': 0.25, 'alpha': 0.40},
        {'pca_dim': 64, 'min_pos': 8,  'C': 0.25, 'alpha': 0.50},
        {'pca_dim': 64, 'min_pos': 8,  'C': 0.50, 'alpha': 0.40},
        {'pca_dim': 64, 'min_pos': 12, 'C': 0.25, 'alpha': 0.40},
        {'pca_dim': 96, 'min_pos': 8,  'C': 0.25, 'alpha': 0.40},
        {'pca_dim': 96, 'min_pos': 8,  'C': 0.50, 'alpha': 0.40},
    ]
    rows = []
    for p in tqdm(param_grid, desc='Probe grid'):
        _scaler = StandardScaler()
        _pca    = PCA(n_components=min(p['pca_dim'], emb_full.shape[0] - 1, emb_full.shape[1]))
        _Z      = _pca.fit_transform(_scaler.fit_transform(emb_full)).astype(np.float32)
        _gkf    = GroupKFold(n_splits=5)
        _groups = meta_full['site'].to_numpy()
        _oof    = oof_base.copy()
        for _, va_idx in _gkf.split(scores_full_raw, groups=_groups):
            tr_idx  = np.setdiff1d(np.arange(len(scores_full_raw)), va_idx)
            pos_cnt = Y_FULL[tr_idx].sum(axis=0)
            for ci in np.where(pos_cnt >= p['min_pos'])[0]:
                y_tr = Y_FULL[tr_idx, ci]
                if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                    continue
                X_tr = build_class_features(
                    _Z[tr_idx], scores_full_raw[tr_idx, ci],
                    oof_prior[tr_idx, ci], oof_base[tr_idx, ci],
                )
                X_va = build_class_features(
                    _Z[va_idx], scores_full_raw[va_idx, ci],
                    oof_prior[va_idx, ci], oof_base[va_idx, ci],
                )
                clf = LogisticRegression(
                    C=p['C'], max_iter=400, solver='liblinear', class_weight='balanced'
                )
                clf.fit(X_tr, y_tr)
                pred = clf.decision_function(X_va).astype(np.float32)
                _oof[va_idx, ci] = (
                    (1.0 - p['alpha']) * oof_base[va_idx, ci] + p['alpha'] * pred
                )
        rows.append({**p, 'oof_auc': macro_auc(Y_FULL, _oof)})
    grid_df = pd.DataFrame(rows).sort_values('oof_auc', ascending=False).reset_index(drop=True)
    print(grid_df.to_string(index=False))
else:
    print(f'Using frozen probe params: pca_dim={CFG.PROBE_PCA_DIM} '
          f'min_pos={CFG.PROBE_MIN_POS} C={CFG.PROBE_C} alpha={CFG.PROBE_ALPHA}')

In [ ]:
pos_counts   = Y_FULL.sum(axis=0)
probe_idx    = np.where(pos_counts >= CFG.PROBE_MIN_POS)[0].astype(np.int32)
probe_models = {}

for cls_idx in tqdm(probe_idx, desc='Training probes'):
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y):
        continue
    X = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=oof_prior[:, cls_idx],
        base_col=oof_base[:, cls_idx],
    )
    clf = LogisticRegression(
        C=CFG.PROBE_C, max_iter=400, solver='liblinear', class_weight='balanced'
    )
    clf.fit(X, y)
    probe_models[cls_idx] = clf

print(f'Probe models trained : {len(probe_models)} / {N_CLASSES} classes')

---

# Test Inference (Perch)

In [ ]:
final_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)

test_paths = sorted((CFG.BASE / 'test_soundscapes').glob('*.ogg'))
if len(test_paths) == 0:
    print(f'No test soundscapes found. Dry-run on {CFG.DRYRUN_N_FILES} train files.')
    test_paths = sorted((CFG.BASE / 'train_soundscapes').glob('*.ogg'))[:CFG.DRYRUN_N_FILES]
else:
    print(f'Test files : {len(test_paths)}')

meta_test, scores_test_raw, emb_test = infer_perch_batch(test_paths)

test_base, test_prior = fuse_scores(
    scores_test_raw,
    meta_test['site'].to_numpy(),
    meta_test['hour_utc'].to_numpy(),
    final_tables,
)

Z_TEST = emb_pca.transform(emb_scaler.transform(emb_test)).astype(np.float32)

final_scores = test_base.copy()
for cls_idx, clf in tqdm(probe_models.items(), desc='Applying probes'):
    X = build_class_features(
        Z_TEST,
        raw_col=scores_test_raw[:, cls_idx],
        prior_col=test_prior[:, cls_idx],
        base_col=test_base[:, cls_idx],
    )
    pred = clf.decision_function(X).astype(np.float32)
    final_scores[:, cls_idx] = (
        (1.0 - CFG.PROBE_ALPHA) * test_base[:, cls_idx]
        + CFG.PROBE_ALPHA * pred
    )

print(f'Perch final_scores : {final_scores.shape}')
print(f'Score range        : {final_scores.min():.3f} to {final_scores.max():.3f}')

# Gaussian smoothing
def gauss_smooth_final(scores, weights=np.array([0.1, 0.2, 0.4, 0.2, 0.1])):
    smoothed = scores.reshape(-1, CFG.N_WINDOWS, scores.shape[1]).copy()
    for i in range(smoothed.shape[0]):
        smoothed[i] = convolve1d(smoothed[i], weights, axis=0, mode='nearest')
    return smoothed.reshape(-1, scores.shape[1])

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

final_scores_smoothed = gauss_smooth_final(final_scores)
perch_probs = sigmoid(final_scores_smoothed).astype(np.float32)
print(f'perch_probs : {perch_probs.shape}')

# Perch 推論後のメモリ解放
del emb_test, scores_test_raw, test_base, test_prior, Z_TEST
del emb_full, scores_full_raw
gc.collect()
print('Perch inference done. Memory released.')

---

# EfficientNet Inference (exp002)

In [ ]:
# Model definition (same as exp002 train.ipynb)
# ── モデル定義（exp002 train.ipynb と同一）────────────────────
class AttBlockV2(nn.Module):
    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.att = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)

    def forward(self, x):
        att = torch.softmax(torch.tanh(self.att(x)), dim=-1)
        cla = self.cla(x)
        return (att * cla).sum(dim=-1)


class BirdCLEFSED(nn.Module):
    def __init__(self, num_classes: int = 234):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnet_b0_ns', pretrained=False,
            in_chans=1, num_classes=0, global_pool='',
        )
        in_features    = self.backbone.num_features
        self.bn        = nn.BatchNorm1d(in_features)
        self.dropout   = nn.Dropout(p=0.3)
        self.att_block = AttBlockV2(in_features, num_classes)

    def forward(self, x):
        feat = self.backbone.forward_features(x)  # (B, C, H, W)
        feat = feat.mean(dim=2)                   # freq pool → (B, C, W)
        feat = self.bn(feat)
        feat = self.dropout(feat)
        return self.att_block(feat)               # (B, num_classes)

In [ ]:
# Mel spectrogram transform
# ── Mel 変換 ──────────────────────────────────────────────────
ell_mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate = CFG.EFF_SAMPLE_RATE,
        n_fft       = CFG.EFF_N_FFT,
        hop_length  = CFG.EFF_HOP_LENGTH,
        n_mels      = CFG.EFF_N_MELS,
        f_min       = CFG.EFF_FMIN,
        f_max       = CFG.EFF_FMAX,
    ),
    T.AmplitudeToDB(top_db=80),
)

def audio_to_melspec(audio_tensor: torch.Tensor) -> torch.Tensor:
    """(batch, n_samples) -> (batch, 1, n_mels, time)"""
    with torch.no_grad():
        mel = ell_mel_transform(audio_tensor)  # (B, n_mels, T)
    mel = mel - mel.amin(dim=(-2, -1), keepdim=True)
    mel = mel / (mel.amax(dim=(-2, -1), keepdim=True) + 1e-8)
    return mel.unsqueeze(1)                    # (B, 1, n_mels, T)


# ── チャンク切り出し（重複推論）───────────────────────────────
def extract_chunk(audio: torch.Tensor, center_sec: float) -> torch.Tensor:
    n, sr = CFG.EFF_N_SAMPLES, CFG.EFF_SAMPLE_RATE
    center = int(center_sec * sr)
    start  = center - n // 2
    end    = start + n
    if start < 0:
        chunk = torch.cat([torch.zeros(-start), audio[:end]])
    elif end > len(audio):
        pad   = end - len(audio)
        chunk = torch.cat([audio[start:], torch.zeros(pad)])
    else:
        chunk = audio[start:end]
    if len(chunk) < n:
        chunk = torch.cat([chunk, torch.zeros(n - len(chunk))])
    return chunk[:n]

In [ ]:
# Load EfficientNet model
# ── モデルロード ──────────────────────────────────────────────
eff_preds_by_rowid = {}

if CFG.EFF_WEIGHT_PATH is not None and CFG.EFF_WEIGHT_PATH.exists():
    checkpoint  = torch.load(str(CFG.EFF_WEIGHT_PATH), map_location='cpu')
    eff_model   = BirdCLEFSED(num_classes=len(checkpoint['labels']))
    eff_model.load_state_dict(checkpoint['model_state_dict'])
    eff_model.eval()
    EFF_LABELS = checkpoint['labels']
    print(f'EfficientNet loaded: epoch={checkpoint["epoch"]}, AUC={checkpoint["best_auc"]:.4f}')
    print(f'Classes: {len(EFF_LABELS)}')

    n_offsets = len(CFG.EFF_OVERLAP_OFFSETS)

    for test_path in tqdm(test_paths, desc='EfficientNet'):
        waveform, sr = torchaudio.load(str(test_path))
        if sr != CFG.EFF_SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sr, CFG.EFF_SAMPLE_RATE)
        audio = waveform.mean(dim=0)  # モノラル化

        # 5秒刻みの end_sec ごとに推論
        for end_sec in range(5, 65, 5):
            center_base = end_sec - 2.5
            chunks = torch.stack([
                extract_chunk(audio, center_base + off)
                for off in CFG.EFF_OVERLAP_OFFSETS
            ])  # (n_offsets, n_samples)

            chunk_preds = []
            for i in range(0, len(chunks), CFG.EFF_BATCH_SIZE):
                specs = audio_to_melspec(chunks[i:i + CFG.EFF_BATCH_SIZE])
                with torch.no_grad():
                    logits = eff_model(specs)
                    probs  = torch.sigmoid(logits).numpy()
                chunk_preds.append(probs)

            seg_preds = np.concatenate(chunk_preds, axis=0).mean(axis=0)  # (num_classes,)
            row_id = f'{test_path.stem}_{end_sec}'
            eff_preds_by_rowid[row_id] = seg_preds

    del eff_model, checkpoint
    gc.collect()
    print(f'EfficientNet inference done. rows={len(eff_preds_by_rowid)}')
else:
    print('WARNING: EfficientNet weight not found. Ensemble will use Perch only (w=1.0).')

---

# Ensemble & Submission

In [ ]:
# Align EfficientNet predictions to Perch row order
# ── EfficientNet 予測を Perch と同じ行順序に整列 ──────────────
test_row_ids = meta_test['row_id'].values

if eff_preds_by_rowid:
    # EFF_LABELS と PRIMARY_LABELS の列順序が合っている前提
    # (どちらも sample_submission.csv の列順に従う)
    eff_matrix = np.array([
        eff_preds_by_rowid.get(rid, np.zeros(N_CLASSES, dtype=np.float32))
        for rid in test_row_ids
    ], dtype=np.float32)
    print(f'eff_matrix shape : {eff_matrix.shape}')

    ensemble_probs = (
        CFG.ENSEMBLE_W_PERCH * perch_probs
        + CFG.ENSEMBLE_W_EFF  * eff_matrix
    ).astype(np.float32)
else:
    # EfficientNet 重みなし → Perch のみ
    ensemble_probs = perch_probs

print(f'ensemble_probs shape : {ensemble_probs.shape}')
print(f'Prob range : {ensemble_probs.min():.4f} to {ensemble_probs.max():.4f}')

In [ ]:
submission = pd.DataFrame(ensemble_probs, columns=PRIMARY_LABELS)
submission.insert(0, 'row_id', test_row_ids)
submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)

assert len(submission) == len(test_paths) * CFG.N_WINDOWS, 'Row count mismatch'
assert submission.columns.tolist() == ['row_id'] + PRIMARY_LABELS, 'Column order mismatch'
assert not submission.isna().any().any(), 'NaNs detected in submission'
assert (submission[PRIMARY_LABELS] >= 0).all().all(), 'Negative probabilities'
assert (submission[PRIMARY_LABELS] <= 1).all().all(), 'Probabilities > 1'

submission.to_csv('submission.csv', index=False)
print('submission.csv saved')
print(f'Shape : {submission.shape}')
submission.iloc[:3, :8]